# 02 – Markdown-Lektorat

Dieses Notebook liegt direkt im Projekt-Root und setzt auf das von der OCR-Pipeline erzeugte Markdown unter `data/processed/<BUCH>/<BUCH>.md` auf.

Die eigenen Lektoratsmodule liegen unter `python/markdown_lektorat/`. Sie werden ohne Installation über einen lokalen `sys.path`-Eintrag geladen. Ein `pyproject.toml` oder `pip install -e .` ist dafür nicht erforderlich.


In [ ]:
from pathlib import Path
import sys

# Projekt-Root finden. Das Notebook selbst liegt im Root; die Suche macht
# die Zelle zusätzlich robust, falls Jupyter mit einem anderen cwd gestartet wurde.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "python" / "markdown_lektorat").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Projekt-Root nicht gefunden: python/markdown_lektorat fehlt.")

PYTHON_DIR = PROJECT_ROOT / "python"
python_dir_str = str(PYTHON_DIR)
if python_dir_str not in sys.path:
    sys.path.insert(0, python_dir_str)

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

# --- Anzupassen
BUCH = None
LMS = "http://192.168.178.27:1234/v1"
MODELL = "google/gemma-4-12b"

print("Projekt-Root      :", PROJECT_ROOT)
print("Python-Module     :", PYTHON_DIR)
print("Arbeitsverzeichnis:", Path.cwd())
print("Python            :", sys.version.split()[0], "\n")

gefunden = (
    sorted(p.stem for p in RAW.iterdir() if p.is_file() and p.suffix.lower() == ".pdf")
    if RAW.exists() else []
)

if BUCH is None:
    if len(gefunden) == 1:
        BUCH = gefunden[0]
        print(f"BUCH automatisch auf {BUCH!r} gesetzt.")
    elif gefunden:
        BUCH = gefunden[0]
        print(f"! {len(gefunden)} Dokumente gefunden – BUCH oben ggf. von Hand setzen.")
    else:
        BUCH = "Buch"
        print("! Kein PDF unter data/raw gefunden – BUCH bleibt Platzhalter.")

MD = PROCESSED / BUCH / f"{BUCH}.md"
print("Markdown:", MD, "->", "vorhanden" if MD.exists() else "fehlt")


In [ ]:
# Lokale Lektoratsmodule laden. autoreload übernimmt Änderungen an den .py-Dateien.
%load_ext autoreload
%autoreload 2

from markdown_lektorat import lekt_config, lekt_pfade, lekt_schema, lekt_markdown
from markdown_lektorat import lekt_schutz, lekt_llm, lekt_checkpoint, lekt_lauf

from markdown_lektorat.lekt_config import (
    AppConfig, InputConfig, LMStudioConfig,
    ConfidenceConfig, ThresholdPolicy, ProcessingConfig, OutputConfig,
)

print("Lektorat-Module geladen aus:", Path(lekt_lauf.__file__).resolve().parent)


In [ ]:
config = AppConfig(
    input=InputConfig(document_path=MD),
    lm=LMStudioConfig(
        base_url=LMS,
        model=MODELL,
        timeout=300.0,
        max_output_tokens=4096,
    ),
    processing=ProcessingConfig(
        max_context_tokens=21000,
        context_reserve_tokens=2500,
        text_chunk_tokens=2000,
        table_chunk_tokens=2500,
        structure_chunk_tokens=2500,
        caption_chunk_tokens=2000,
        overlap_blocks=2,
        retry_count=2,
        resume=True,
    ),
    confidence=ConfidenceConfig(default=ThresholdPolicy()),
    output=OutputConfig(overwrite=False),
)
config


## Dry Run – nur Struktur prüfen, noch kein LLM-Aufruf


In [ ]:
text = MD.read_text(encoding="utf-8", errors="strict")
dok = lekt_markdown.parse_markdown(text)
geschuetzt = lekt_schutz.inventory_protected(text)

print("Blöcke        :", len(dok.blocks))
print("Tabellen      :", sum(b.block_type == "table" for b in dok.blocks))
print("Bilder        :", len(geschuetzt.images))
print("Seitenumbrüche:", len(geschuetzt.pagebreaks))


## LM Studio prüfen


In [ ]:
CLIENT = lekt_llm.LMStudioOpenAIClient(
    config.lm,
    retry_count=config.processing.retry_count,
)
CLIENT.preflight()
print("LM Studio erreichbar; Modell verfügbar:", config.lm.model)


## Korrekturlauf


In [ ]:
# Vor dem Lauf: vorhandenen Checkpoint anzeigen.
CP_DIR = lekt_lauf._checkpoint_dir(MD)
CP = CP_DIR / "checkpoint.json"
if CP.exists():
    state = lekt_checkpoint.load_checkpoint(CP)
    print("Fortsetzbarer Lauf gefunden:")
    print("  Status       :", state.status)
    print("  Pass         :", state.current_pass)
    print("  Nächster Chunk:", f"{state.current_pass}:{state.next_chunk_index:04d}")
else:
    print("Kein Checkpoint vorhanden – neuer Lauf.")


In [ ]:
ERGEBNIS = lekt_lauf.correct_markdown(config, client=CLIENT, fortschritt=print)

print("Status     :", ERGEBNIS.status)
print("Korrigiert :", ERGEBNIS.corrected_path)
print("Review     :", ERGEBNIS.review_path)
print("Manifest   :", ERGEBNIS.manifest_path)
print("Einträge   :", len(ERGEBNIS.entries))
print("Unresolved :", sum(e.status == "unresolved" for e in ERGEBNIS.entries))
